# Выборка для обнаружения взрослых бабочек
Откройте ноутбук в Jupyter или VS Code с ядром Python 3.9+ и выполните все ячейки. При TARGET = 400 план сбора — 360 наблюдений взрослых чешуекрылых, 20 наблюдений личинок или куколок чешуекрылых и 20 наблюдений других насекомых. Один вид представлен максимум одним фото во всей выборке. Если группа исчерпана, её недобор виден в отчёте.

Результат — одна таблица `sources.tsv` с шестью полями. Изображения не сохраняются на диск, но загружаются в память для проверки читаемости и дедупликации по хешу декодированных пикселей. Хеши хранятся только в памяти и не попадают в таблицу.

`is_butterfly = 1` для группы взрослых чешуекрылых (включая мотыльков), `0` для личинок, куколок и других насекомых. Метка основана на фильтрах наблюдения iNaturalist, а не на анализе изображения: отдельные кадры могут не соответствовать аннотации наблюдения.

Для пробного запуска можно поставить TARGET = 20. Каждый запуск собирает выборку заново и перезаписывает таблицу; строки сохраняются по мере сбора и перемешиваются с фиксированным seed. Запускайте один экземпляр ноутбука на папку. Файлы от прежних запусков автоматически не удаляются.


In [9]:
%pip install Pillow tqdm ipywidgets

/Users/anrewqz/учеба/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path

TARGET = 420
OUT = Path("inat_butterflies_mixed_cc0").resolve()
print("Папка результата:", OUT)

Папка результата: /Users/anrewqz/учеба/краудсорсинг/проект/inat_butterflies_mixed_cc0


In [11]:
import csv
import hashlib
import io
import json
import time
import random
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit, urlunsplit
from urllib.request import Request, urlopen

from PIL import Image, ImageOps
from tqdm.auto import tqdm

API = "https://api.inaturalist.org/v1/observations"
FILTERS = dict(photo_license="cc0",
               photos="true", quality_grade="research", rank="species",
               order_by="id", order="desc", per_page=200)
FIELDS = ["observation_id", "photo_id", "taxon_id",
          "observation_url", "photo_url", "is_butterfly"]

def fetch(url):
    for attempt in range(5):
        try:
            req = Request(url, headers={"User-Agent": "ButterflyDatasetNotebook/1.0"})
            with urlopen(req, timeout=45) as response:
                return response.read()
        except HTTPError as exc:
            if exc.code not in (429, 500, 502, 503, 504) or attempt == 4:
                raise
            retry_after = exc.headers.get("Retry-After", "")
            delay = int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1)
        except (URLError, TimeoutError, ConnectionError):
            if attempt == 4:
                raise
            delay = 2 ** (attempt + 1)
        time.sleep(max(1, delay))

def large_url(url):
    parts = urlsplit(url)
    directory, _, filename = parts.path.rpartition("/")
    if parts.scheme != "https" or not filename.startswith("square."):
        return None
    return urlunsplit(parts._replace(path=directory + "/large." + filename.split(".", 1)[1]))

def inspect_image(data):
    with Image.open(io.BytesIO(data)) as image:
        ext = {"JPEG": "jpg", "PNG": "png", "WEBP": "webp"}.get(image.format)
        if not ext:
            raise ValueError("Неподдерживаемый формат изображения")
        image.load()
        rgb = ImageOps.exif_transpose(image).convert("RGB")
        digest = hashlib.sha256(str(rgb.size).encode("ascii"))
        digest.update(rgb.tobytes())
    return digest.hexdigest()

def write_table(name, fields, rows):
    dest = OUT / name
    temp = dest.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields, delimiter="\t", extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(dest)

def checkpoint(rows):
    rows = list(rows)
    random.Random(42).shuffle(rows)
    write_table("sources.tsv", FIELDS, rows)

def get_photo(obs, photos_used, pixels_used, is_butterfly):
    for photo in obs.get("photos", []):
        pid = str(photo["id"])
        if str(photo.get("license_code")).lower() != "cc0" or pid in photos_used:
            continue
        url = large_url(photo.get("url") or "")
        if not url:
            continue
        try:
            data = fetch(url)
            pixels = inspect_image(data)
            del data
        except HTTPError as exc:
            if exc.code not in (403, 404, 410):
                raise
            continue
        except (URLError, TimeoutError, ConnectionError):
            raise
        except (OSError, ValueError, Image.DecompressionBombError) as exc:
            tqdm.write(f"Пропуск нечитаемого фото {pid}: {exc}")
            continue
        if pixels in pixels_used:
            continue
        row = {"observation_id": str(obs["id"]), "photo_id": pid,
               "taxon_id": str(obs["taxon"]["id"]),
               "observation_url": f"https://www.inaturalist.org/observations/{obs['id']}",
               "photo_url": url,
               "is_butterfly": is_butterfly}
        return row, pixels


In [12]:
def collect():
    if not isinstance(TARGET, int) or TARGET < 1:
        raise ValueError("TARGET должен быть положительным целым числом")
    OUT.mkdir(parents=True, exist_ok=True)
    rows = []
    checkpoint(rows)
    species = {row["taxon_id"] for row in rows}
    photos = {row["photo_id"] for row in rows}
    pixels = set()
    side_quota = TARGET // 20
    groups = [
        ("adult_lepidoptera", TARGET - 2 * side_quota,
         dict(taxon_id=47157, term_id=1, term_value_id=2)),
        ("larva_or_pupa", side_quota, dict(taxon_id=47157, term_id=1, term_value_id="4,6")),
        ("other_insects", side_quota, dict(taxon_id=47158, without_taxon_id=47157)),
    ]
    with tqdm(total=TARGET, desc="Загрузка и отбор фото", unit="фото") as progress:
        for group, quota, filters in groups:
            progress.set_postfix_str(group)
            count = 0
            is_butterfly = int(group == "adult_lepidoptera")
            cursor = None
            while count < quota:
                params = {**FILTERS, **filters}
                if cursor is not None:
                    params["id_below"] = cursor
                time.sleep(1.1)
                observations = json.loads(fetch(API + "?" + urlencode(params)))["results"]
                if not observations:
                    tqdm.write("Доступные наблюдения исчерпаны.")
                    break
                next_cursor = min(o["id"] for o in observations)
                if cursor is not None and next_cursor >= cursor:
                    raise RuntimeError("API вернул повторную страницу. Результаты сохранены.")
                cursor = next_cursor
                for obs in observations:
                    taxon = obs.get("taxon") or {}
                    if taxon.get("rank") != "species" or str(taxon["id"]) in species:
                        continue
                    result = get_photo(obs, photos, pixels, is_butterfly)
                    if result is None:
                        continue
                    row, pixel_hash = result
                    rows.append(row)
                    count += 1
                    checkpoint(rows)
                    species.add(row["taxon_id"])
                    photos.add(row["photo_id"])
                    pixels.add(pixel_hash)
                    progress.update(1)
                    if count >= quota:
                        break
            tqdm.write(f"{group}: {count}/{quota}")
    print(f"Фото: {len(rows)}. Разных видов: {len(species)}. Цель: {TARGET}.")
    print(OUT / "sources.tsv")

collect()

Загрузка и отбор фото:   5%|▌         | 22/420 [01:49<17:43,  2.67s/фото, adult_lepidoptera] 

Пропуск нечитаемого фото 763915754: The read operation timed out


Загрузка и отбор фото:  90%|█████████ | 378/420 [11:06<00:55,  1.31s/фото, larva_or_pupa]     

adult_lepidoptera: 378/378


Загрузка и отбор фото:  95%|█████████▌| 399/420 [12:05<00:26,  1.26s/фото, other_insects]

larva_or_pupa: 21/21


Загрузка и отбор фото: 100%|██████████| 420/420 [13:06<00:00,  1.87s/фото, other_insects]

other_insects: 21/21
Фото: 420. Разных видов: 420. Цель: 420.
/Users/anrewqz/учеба/краудсорсинг/проект/inat_butterflies_mixed_cc0/sources.tsv


### Результат
`sources.tsv` — единственная создаваемая таблица, в формате TSV (UTF-8):

* `observation_id` — ID наблюдения.
* `photo_id` — ID фотографии.
* `taxon_id` — ID вида.
* `observation_url` — ссылка на страницу наблюдения.
* `photo_url` — прямая HTTPS-ссылка на файл изображения (версия large), без элементов сайта.
* `is_butterfly` — 1 для взрослых чешуекрылых, 0 для личинок, куколок и других насекомых, по группе наблюдения.

Сохраняется отбор фотографий с лицензией CC0. Дубликаты исключаются по ID фотографии, виду и SHA-256 декодированных RGB-пикселей с учётом размера и EXIF-ориентации. Изображения и хеши на диск не записываются. После прерывания частичная таблица остаётся на диске; следующий запуск начинает сбор заново.
